<a href="https://colab.research.google.com/github/AI-is-out-there/2026-REA-Diploma-Practical-Part/blob/dev/FireCrawl-CR-site-parsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install the latest version of Firecrawl
!pip install firecrawl-py --upgrade -q

from firecrawl import Firecrawl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.1/248.1 kB 7.3 MB/s eta 0:00:00


In [ ]:
API_KEY = 'fc-7f29fdf62edf482c9bd1a65cbcb4eb78'

In [ ]:
app = Firecrawl(api_key=API_KEY)

def search_and_extract_with_ai(keyword):
    """
    Uses Firecrawl's new AI 'Interact' feature to control a real browser
    using natural language prompts. This bypasses all complex JavaScript
    and anti-bot protections.
    """
    print(f"🔍 Starting Firecrawl AI session for: '{keyword}'...")

    try:
        # 1. Initial scrape to start the browser session and get a scrape_id
        print("🌐 Opening the clinical guidelines registry...")
        result = app.scrape("https://cr.minzdrav.gov.ru/clin-rec", formats=["markdown"])

        # Extract the session ID (handle both object and dict returns just in case)
        if hasattr(result, 'metadata') and hasattr(result.metadata, 'scrape_id'):
            scrape_id = result.metadata.scrape_id
        elif isinstance(result, dict) and 'metadata' in result:
            scrape_id = result['metadata'].get('scrape_id') or result['metadata'].get('scrapeId')
        else:
            print("❌ Could not find scrape_id in the response.")
            return None

        print(f"✅ Browser session started. ID: {scrape_id}")

        # 2. AI Action: Type the keyword and search
        print(f"🔎 AI Agent is typing '{keyword}' into the search box...")
        app.interact(
            scrape_id,
            prompt=f"Find the search input field on this page, type '{keyword}' into it, and submit the search (press Enter or click the search button)."
        )

        # 3. AI Action: Click the first result
        print("📄 AI Agent is clicking the first search result...")
        app.interact(
            scrape_id,
            prompt="Wait for the search results to load. Then, click on the very first link in the search results list to open the Clinical Recommendation page."
        )

        # 4. AI Action: Navigate to Appendix B and extract the text
        print("📑 AI Agent is finding and extracting 'Doctor's Action Algorithms' (Appendix B)...")
        response = app.interact(
            scrape_id,
            prompt=(
                "Find the section titled 'Приложение Б' or 'Алгоритмы действий врача' (Doctor's Action Algorithms). "
                "Click on it to expand it if it is collapsed. "
                "Then, extract and return the full text content of this specific section. "
                "Do not summarize it, return the exact text."
            )
        )

        # The AI agent returns the extracted text in the 'output' field
        extracted_text = response.output
        print("✅ Successfully extracted the Doctor's Actions Algorithm!")

        # 5. Clean up the browser session to save credits
        app.stop_interaction(scrape_id)
        print("🧹 Browser session closed.")

        return extracted_text

    except Exception as e:
        print(f"❌ An error occurred: {e}")
        return None

# --- RUN THE SCRIPT ---
search_keyword = "Прогрессирующая мышечная дистрофия"
result = search_and_extract_with_ai(search_keyword)

if result:
    print("\n" + "="*60)
    print("EXTRACTED DOCTOR ACTIONS ALGORITHM:")
    print("="*60)
    print(result)
else:
    print("Failed to extract data.")

🔍 Starting Firecrawl AI session for: 'Прогрессирующая мышечная дистрофия'...
🌐 Opening the clinical guidelines registry...
✅ Browser session started. ID: 019eff41-892e-707a-9f4e-649a2698a439
🔎 AI Agent is typing 'Прогрессирующая мышечная дистрофия' into the search box...
📄 AI Agent is clicking the first search result...
📑 AI Agent is finding and extracting 'Doctor's Action Algorithms' (Appendix B)...
✅ Successfully extracted the Doctor's Actions Algorithm!
🧹 Browser session closed.

EXTRACTED DOCTOR ACTIONS ALGORITHM:
The requested sections "Приложение Б" or "Алгоритмы действий врача" are not present on the current page. The page indicates that technical work is being performed.


# Part II

In [13]:
from firecrawl import Firecrawl
import re

API_KEY = 'fc-7f29fdf62edf482c9bd1a65cbcb4eb78'
app = Firecrawl(api_key=API_KEY)


def search_and_extract_with_ai(keyword: str):
    """
    AI-powered RPA workflow:
    1. Open https://cr.minzdrav.gov.ru/clin-rec
    2. Type keyword into search field under "Наименование"
    3. Click first result row → get CR ID *directly from the 'ID' column cell*
    4. Navigate to /preview-cr/{ID}#doc_b
    5. Extract full text of "Приложение Б"

    ID is extracted as raw string (preserving underscores, hyphens, prefixes).
    """
    print(f"\n🔍 Starting Firecrawl AI session for: '{keyword}'")

    try:
        # 1. Open registry
        print("🌐 Opening clinical guidelines registry...")
        result = app.scrape("https://cr.minzdrav.gov.ru/clin-rec", formats=["markdown"])

        # Extract scrape_id robustly
        scrape_id = None
        if hasattr(result, 'metadata') and hasattr(result.metadata, 'scrape_id'):
            scrape_id = result.metadata.scrape_id
        elif isinstance(result, dict) and 'metadata' in result:
            scrape_id = result['metadata'].get('scrape_id') or result['metadata'].get('scrapeId')
        if not scrape_id:
            raise RuntimeError("❌ No scrape_id in initial response")

        print(f"✅ Browser session started. ID: {scrape_id}")

        # 2. AI: Type into search field under "Наименование"
        print(f"🔎 AI Agent typing '{keyword}' into the 'Наименование' filter...")
        app.interact(
            scrape_id,
            prompt=f"Locate the search input under the table header 'Наименование', type '{keyword}', and press Enter to filter."
        )

        # 3. AI: Click first result row → then extract ID from the *ID column cell* (leftmost)
        print("🆔 AI Agent extracting CR ID from first row's 'ID' column (raw string)...")
        id_response = app.interact(
            scrape_id,
            prompt=("""\
                The filtered table is loaded. Locate the first data row.
                Find the cell directly under the column header 'ID' (typically the leftmost column).
                Extract the *exact text content* of that cell — it may contain digits, underscores, hyphens, or letters (e.g., '230_3', '145').
                Return ONLY that raw string, no extra characters, no explanations.
                """
            )
        )
        raw_id = id_response.output.strip()

        # Sanitize: remove common noise (quotes, brackets, surrounding whitespace)
        raw_id = re.sub(r'^[\'"`\[\{]+|[\'"`\]\}]+$', '', raw_id)  # strip quotes/brackets
        cr_id = raw_id.strip()
        if not cr_id:
            raise ValueError("Empty ID extracted after cleaning")

        print(f"✅ Extracted CR ID (raw): '{cr_id}'")

        # 4. Navigate to Appendix B
        preview_url = f"https://cr.minzdrav.gov.ru/preview-cr/{cr_id}#doc_b"
        print(f"➡️ Loading: {preview_url}")
        _ = app.scrape(preview_url, formats=["markdown"])  # ensure anchor loads

        # 5. AI: Extract full text of "Приложение Б"
        print("📑 AI Agent extracting 'Приложение Б' (Doctor's Action Algorithm)...")
        algo_response = app.interact(
            scrape_id,
            prompt=(
                "Find the section titled 'Приложение Б' or 'Алгоритмы действий врача'. "
                "If collapsed, expand it. Then select and return the FULL visible text content — verbatim, no summarization. "
                "Only return the raw text."
            )
        )
        extracted_text = algo_response.output.strip()
        if not extracted_text:
            raise RuntimeError("❌ Empty algorithm text returned")

        print("✅ Successfully extracted Doctor's Action Algorithm!")

        # 6. Cleanup
        app.stop_interaction(scrape_id)
        print("🧹 Browser session closed.")

        return {
            "original_input": keyword,
            "cr_id": cr_id,
            "algorithm_text": extracted_text,
            "status": "success"
        }

    except Exception as e:
        print(f"❌ Error: {type(e).__name__}: {e}")
        if 'scrape_id' in locals():
            try:
                app.stop_interaction(scrape_id)
            except:
                pass
        return {
            "original_input": keyword,
            "error": str(e),
            "status": "failed"
        }


# === MAIN EXECUTION ===
if __name__ == "__main__":
    kw = input("Enter CR title (e.g., 'Прогрессирующая мышечная дистрофия'): ").strip()
    if not kw:
        print("⚠️ Empty input. Exiting.")
        exit(1)

    res = search_and_extract_with_ai(kw)

    print("\n" + "="*80)
    print("FIRECRAWL CR EXTRACTION RESULT")
    print("="*80)
    print(f"Input:          {res['original_input']}")
    if res['status'] == 'success':
        print(f"CR ID (raw):    '{res['cr_id']}'")  # preserves format: 230_3, 145-7, etc.
        print(f"Algorithm len:  {len(res['algorithm_text'])} chars")
        print("\nFirst 300 chars of Algorithm:")
        print(repr(res['algorithm_text'][:300] + ("..." if len(res['algorithm_text']) > 300 else "")))
    else:
        print(f"Error:          {res['error']}")
    print("="*80)

Enter CR title (e.g., 'Прогрессирующая мышечная дистрофия'): Болезнь Грейвса

🔍 Starting Firecrawl AI session for: 'Болезнь Грейвса'
🌐 Opening clinical guidelines registry...
✅ Browser session started. ID: 019f03ab-75e6-75eb-832d-8adf8761f478
🔎 AI Agent typing 'Болезнь Грейвса' into the 'Наименование' filter...
🆔 AI Agent extracting CR ID from first row's 'ID' column (raw string)...
✅ Extracted CR ID (raw): '270_3'
➡️ Loading: https://cr.minzdrav.gov.ru/preview-cr/270_3#doc_b
📑 AI Agent extracting 'Приложение Б' (Doctor's Action Algorithm)...
✅ Successfully extracted Doctor's Action Algorithm!
🧹 Browser session closed.

FIRECRAWL CR EXTRACTION RESULT
Input:          Болезнь Грейвса
CR ID (raw):    '270_3'
Algorithm len:  227 chars

First 300 chars of Algorithm:
'Алгоритмы действий врача\nГлавная\n/\nАлгоритмы действия врача\nАлгоритмы действий врача\nID\tНаименование\n\n\t\n\n569_2\tЖелудочковые нарушения ритма сердца. Внезапная сердечная смерть.\n574_1\tРак желудка\nСтрок на странице: